# Extra Option B - Scoring Profile in Azure AI Search

## Objective
Add a scoring profile to the existing index and show how it changes ranking.

In this notebook I will:
1. Connect to Azure AI Search with `DefaultAzureCredential`.
2. Create or update a scoring profile that boosts important text fields.
3. Run the same query before and after applying the profile.
4. Compare ranking and scores to explain impact.

## 0) Setup

Expected `.env` values:
- `AZURE_SEARCH_ENDPOINT`
- `AZURE_SEARCH_INDEX`

Optional values used in this notebook:
- `SCORING_PROFILE_NAME` (default: `boost-title-profile`)

In [11]:
import os
from typing import Dict, List

import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import ScoringProfile, TextWeights
from azure.core.exceptions import HttpResponseError

# Load notebook-specific environment variables from .env.
load_dotenv(override=True)

AZURE_SEARCH_ENDPOINT = os.getenv('AZURE_SEARCH_ENDPOINT', '').strip()
AZURE_SEARCH_INDEX = os.getenv(
    'AZURE_SEARCH_INDEX', 'rag-1776279755924').strip()
SCORING_PROFILE_NAME = os.getenv(
    'SCORING_PROFILE_NAME', 'boost-title-profile').strip()

if not AZURE_SEARCH_ENDPOINT:
    raise ValueError('Missing AZURE_SEARCH_ENDPOINT in .env')

# DefaultAzureCredential reuses signed-in developer identity (VS Code / Azure CLI / browser).
credential = DefaultAzureCredential(
    exclude_interactive_browser_credential=False)
search_client = SearchClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    index_name=AZURE_SEARCH_INDEX,
    credential=credential,
)
index_client = SearchIndexClient(
    endpoint=AZURE_SEARCH_ENDPOINT, credential=credential)

print('Connected to Azure AI Search')
print(f'Endpoint: {AZURE_SEARCH_ENDPOINT}')
print(f'Index: {AZURE_SEARCH_INDEX}')
print(f'Scoring profile name: {SCORING_PROFILE_NAME}')

Connected to Azure AI Search
Endpoint: https://04-ai-search-resource.search.windows.net
Index: rag-1776279755924
Scoring profile name: boost-title-profile


In [12]:
# Quick connectivity and authorization check on the index.
try:
    probe = search_client.search(
        search_text='*',
        top=1,
        include_total_count=True,
    )
    print(f'Connectivity OK. Document count: {probe.get_count()}')
except HttpResponseError as ex:
    print('Connectivity check failed. Verify endpoint, index name, and RBAC roles.')
    print(ex)

Connectivity OK. Document count: 8


## 1) Create/Update Scoring Profile

You can create the scoring profile in either of these two ways:
- **Azure Portal**: go to your Search index in the portal and configure the scoring profile in the UI (example shown below).
- **Python code**: run the next code cell in this notebook to create/update the same profile programmatically.

![Scoring profile configuration in Azure Portal](images/07_scoring_profile_configuration.png)

This profile boosts matches in `title`, then `chunk`, and lightly weights `chunk_id` to reflect your current index schema.
`text_vector` is intentionally not included in text weights because vector fields are ranked through vector similarity, not lexical scoring profiles.

In [13]:
def build_text_weights(field_names: List[str]) -> Dict[str, float]:
    # Match this assignment index schema and prioritize human-readable relevance.
    candidate_weights = [
        ('title', 5.0),
        ('chunk', 2.0),
        ('chunk_id', 1.1),
    ]
    weights: Dict[str, float] = {}
    for name, weight in candidate_weights:
        if name in field_names:
            weights[name] = weight

    if not weights:
        raise ValueError(
            'No compatible text fields found for scoring profile text_weights.')

    return weights


def create_or_update_scoring_profile() -> None:
    index = index_client.get_index(AZURE_SEARCH_INDEX)
    field_names = [f.name for f in index.fields]
    weights = build_text_weights(field_names)

    # Note: text_vector is used for vector similarity, not text_weights scoring.
    if 'text_vector' in field_names:
        print('Info: text_vector detected and excluded from text_weights.')

    profile = ScoringProfile(
        name=SCORING_PROFILE_NAME,
        text_weights=TextWeights(weights=weights),
    )

    existing_profiles = list(index.scoring_profiles or [])
    profile_names = [p.name for p in existing_profiles]

    if SCORING_PROFILE_NAME in profile_names:
        existing_profiles = [
            p for p in existing_profiles if p.name != SCORING_PROFILE_NAME]

    existing_profiles.append(profile)
    index.scoring_profiles = existing_profiles

    # Keep default scoring profile empty so baseline queries stay truly neutral.
    index.default_scoring_profile = None

    index_client.create_or_update_index(index)
    print(f'Scoring profile configured: {SCORING_PROFILE_NAME}')
    print(f'Text weights: {weights}')
    print('Default scoring profile cleared for fair baseline comparison.')


create_or_update_scoring_profile()

Info: text_vector detected and excluded from text_weights.
Scoring profile configured: boost-title-profile
Text weights: {'title': 5.0, 'chunk': 2.0, 'chunk_id': 1.1}
Default scoring profile cleared for fair baseline comparison.


## 2) Run Same Query Without vs With Scoring Profile

We run the same query twice so ranking changes are easy to observe.

In [ ]:
QUERY_TEXT = 'hybrid search and vector search differences'
TOP_K = 5

# Build a safe select list from fields that actually exist in the index.
index_definition = index_client.get_index(AZURE_SEARCH_INDEX)
available_fields = {f.name for f in index_definition.fields}
preferred_fields = ['chunk_id', 'title', 'chunk', 'parent_id']
SELECT_FIELDS = [name for name in preferred_fields if name in available_fields]

if not SELECT_FIELDS:
    raise ValueError(
        'No compatible fields available for select in this index.')


def to_df(results, run_label: str) -> pd.DataFrame:
    rows = []
    for rank, item in enumerate(results, start=1):
        rows.append({
            'run': run_label,
            'rank': rank,
            'title': item.get('title', ''),
            'chunk_id': item.get('chunk_id', ''),
            'score': item.get('@search.score'),
        })
    return pd.DataFrame(rows)


print('Running baseline query (no scoring_profile argument).')
baseline = search_client.search(
    search_text=QUERY_TEXT,
    top=TOP_K,
    select=SELECT_FIELDS,
    include_total_count=True,
)
df_baseline = to_df(baseline, 'Without scoring profile')

print(f'Running query with scoring_profile={SCORING_PROFILE_NAME!r}.')
with_profile = search_client.search(
    search_text=QUERY_TEXT,
    top=TOP_K,
    select=SELECT_FIELDS,
    include_total_count=True,
    scoring_profile=SCORING_PROFILE_NAME,
)
df_profile = to_df(with_profile, 'With scoring profile')

display(df_baseline)
display(df_profile)

# Compact diagnostic summary to quickly detect score/rank impact.
baseline_top = df_baseline.iloc[0]['title'] if not df_baseline.empty else 'n/a'
profile_top = df_profile.iloc[0]['title'] if not df_profile.empty else 'n/a'
baseline_score = float(
    df_baseline.iloc[0]['score']) if not df_baseline.empty else 0.0
profile_score = float(
    df_profile.iloc[0]['score']) if not df_profile.empty else 0.0
score_ratio = (profile_score /
               baseline_score) if baseline_score else float('nan')

print(f'Top title without profile: {baseline_top}')
print(f'Top title with profile: {profile_top}')
print(f'Top score ratio (with/without): {score_ratio:.3f}')

Running baseline query (no scoring_profile argument).
Running query with scoring_profile='boost-title-profile'.


,run,rank,title,chunk_id,score
0,Without scoring profile,1,08_quick_faq.html,17c0b3c635c2_aHR0cHM6Ly9zdHJncmVzb3VyY2UuYmxvY...,1.706504
1,Without scoring profile,2,06_hybrid_search_playbook.html,17c0b3c635c2_aHR0cHM6Ly9zdHJncmVzb3VyY2UuYmxvY...,1.519549
2,Without scoring profile,3,01_rag_pipeline_overview.html,17c0b3c635c2_aHR0cHM6Ly9zdHJncmVzb3VyY2UuYmxvY...,1.509178
3,Without scoring profile,4,03_semantic_configuration_notes.html,17c0b3c635c2_aHR0cHM6Ly9zdHJncmVzb3VyY2UuYmxvY...,1.428561
4,Without scoring profile,5,04_vector_profile_and_algorithm.html,17c0b3c635c2_aHR0cHM6Ly9zdHJncmVzb3VyY2UuYmxvY...,0.941617


,run,rank,title,chunk_id,score
0,With scoring profile,1,08_quick_faq.html,17c0b3c635c2_aHR0cHM6Ly9zdHJncmVzb3VyY2UuYmxvY...,3.413008
1,With scoring profile,2,06_hybrid_search_playbook.html,17c0b3c635c2_aHR0cHM6Ly9zdHJncmVzb3VyY2UuYmxvY...,3.039098
2,With scoring profile,3,01_rag_pipeline_overview.html,17c0b3c635c2_aHR0cHM6Ly9zdHJncmVzb3VyY2UuYmxvY...,3.018357
3,With scoring profile,4,03_semantic_configuration_notes.html,17c0b3c635c2_aHR0cHM6Ly9zdHJncmVzb3VyY2UuYmxvY...,2.857121
4,With scoring profile,5,04_vector_profile_and_algorithm.html,17c0b3c635c2_aHR0cHM6Ly9zdHJncmVzb3VyY2UuYmxvY...,1.883234


Top title without profile: 08_quick_faq.html
Top title with profile: 08_quick_faq.html
Top score ratio (with/without): 2.000


## 3) Brief Impact Analysis

Based on the observed output in this run:

- The top-5 ranking order stayed the same before and after applying the scoring profile.
- The scores increased clearly with the scoring profile (roughly about 2x in the top results).
- This means the profile is active, but for this specific query the boost affected candidates similarly, so ordering did not change.

My conclusion for this experiment:
- Main ranking change: no reorder in top-5, but stronger scoring signal.
- Reason: boosted fields (`title`, `chunk`) appear to be relevant across the same leading documents.
- Recommendation: keep this profile for stronger lexical relevance, and test with broader queries to reveal larger rank shifts.